In [1]:
import json
import re

In [4]:
def fix_latex(input_file, output_file):
    """
    Fix LaTeX wrapping in JSONL file.
    This version first removes incorrectly placed $ signs, then re-wraps properly.
    """
    
    latex_commands = [
        'alpha', 'beta', 'gamma', 'delta', 'epsilon', 'zeta', 'eta', 'theta',
        'iota', 'kappa', 'lambda', 'mu', 'nu', 'xi', 'pi', 'rho', 'sigma',
        'tau', 'upsilon', 'phi', 'chi', 'psi', 'omega',
        'Gamma', 'Delta', 'Theta', 'Lambda', 'Xi', 'Pi', 'Sigma', 'Phi', 'Psi', 'Omega',
        'times', 'div', 'pm', 'mp', 'approx', 'neq', 'leq', 'geq', 'infty',
        'int', 'sum', 'prod', 'sqrt', 'partial', 'nabla', 'in', 'notin',
        'subset', 'supset', 'cup', 'cap', 'land', 'lor', 'neg',
        'rightarrow', 'leftarrow', 'leftrightarrow',
        'Rightarrow', 'Leftarrow', 'Leftrightarrow',
        'forall', 'exists', 'emptyset', 'perp', 'parallel', 'propto',
        'equiv', 'cong', 'sim', 'circ', 'cdot', 'frac', 'exp', 'log', 'ln',
        'sin', 'cos', 'tan', 'sec', 'csc', 'cot', 'arcsin', 'arccos', 'arctan',
        'sinh', 'cosh', 'tanh', 'lim', 'max', 'min', 'sup', 'inf',
        'det', 'dim', 'ker', 'hom', 'arg', 'deg', 'gcd', 'lcm',
        'binom', 'choose', 'pmod', 'bmod', 'text', 'mathrm', 'mathbf',
        'vec', 'hat', 'bar', 'dot', 'ddot', 'tilde', 'overline', 'underline',
        'overbrace', 'underbrace', 'left', 'right', 'big', 'Big', 'bigg', 'Bigg',
    ]
    
    def strip_dollar_signs(text):
        """Remove all $ signs from text, preserving the content."""
        # Simply remove all $ signs - we'll re-add them properly
        return text.replace('$', '')
    
    def find_matching_brace(text, start):
        """Find the matching closing brace for an opening brace at position start."""
        if start >= len(text) or text[start] != '{':
            return -1
        count = 1
        i = start + 1
        while i < len(text) and count > 0:
            if text[i] == '{':
                count += 1
            elif text[i] == '}':
                count -= 1
            i += 1
        return i - 1 if count == 0 else -1
    
    def wrap_math_expressions(text):
        """Wrap mathematical expressions in $...$."""
        
        # Sort commands by length (longest first) to avoid partial matches
        sorted_cmds = sorted(latex_commands, key=len, reverse=True)
        cmd_pattern = '|'.join(re.escape(cmd) for cmd in sorted_cmds)
        
        # Find all LaTeX commands and their full extent
        result = []
        i = 0
        while i < len(text):
            # Check for LaTeX command
            match = re.match(r'\\(' + cmd_pattern + r')', text[i:])
            if match:
                # Found a LaTeX command, now find its full extent
                start = i
                cmd = match.group(1)
                i += len(match.group(0))
                
                # Consume any following arguments and modifiers
                while i < len(text):
                    # Skip whitespace ONLY if followed by a valid continuation
                    # Save position to backtrack if needed
                    ws_start = i
                    while i < len(text) and text[i] in ' \t':
                        i += 1
                    
                    if i >= len(text):
                        i = ws_start  # backtrack - don't include trailing ws
                        break
                    
                    # Check for braced argument {....}
                    if text[i] == '{':
                        end_brace = find_matching_brace(text, i)
                        if end_brace > i:
                            i = end_brace + 1
                            continue
                        else:
                            i = ws_start  # backtrack
                            break
                    
                    # Check for subscript _ or superscript ^
                    if text[i] in '_^':
                        i += 1
                        if i < len(text):
                            if text[i] == '{':
                                end_brace = find_matching_brace(text, i)
                                if end_brace > i:
                                    i = end_brace + 1
                                    continue
                            elif text[i].isalnum() or text[i] in '+-':
                                i += 1
                                continue
                            elif text[i] == '\\':
                                # Handle ^/_ followed by LaTeX command like \infty
                                # Check if it's a known command
                                cmd_match = re.match(r'\\(' + cmd_pattern + r')', text[i:])
                                if cmd_match:
                                    i += len(cmd_match.group(0))
                                    continue
                        # Fall through to break
                    
                    # No valid continuation found, backtrack whitespace
                    i = ws_start
                    break
                
                # Extract the full expression and wrap it
                expr = text[start:i]
                result.append('$' + expr + '$')
            
            # Check for standalone subscript/superscript on single letter (like T_0, x^2)
            elif i + 1 < len(text) and text[i].isalpha() and text[i+1] in '_^':
                start = i
                i += 1  # skip the letter
                
                # Consume subscripts and superscripts
                while i < len(text) and text[i] in '_^':
                    i += 1
                    if i < len(text):
                        if text[i] == '{':
                            end_brace = find_matching_brace(text, i)
                            if end_brace > i:
                                i = end_brace + 1
                            else:
                                break
                        elif text[i].isalnum() or text[i] in '+-':
                            i += 1
                        else:
                            break
                    else:
                        break
                
                expr = text[start:i]
                result.append('$' + expr + '$')
            
            # Check for e^{...} (exponential)
            elif text[i:i+2] == 'e^' or text[i:i+3] == 'e^{':
                start = i
                i += 1  # skip 'e'
                if i < len(text) and text[i] == '^':
                    i += 1
                    if i < len(text):
                        if text[i] == '{':
                            end_brace = find_matching_brace(text, i)
                            if end_brace > i:
                                i = end_brace + 1
                        elif text[i].isalnum() or text[i] in '+-':
                            i += 1
                
                expr = text[start:i]
                result.append('$' + expr + '$')
            
            else:
                result.append(text[i])
                i += 1
        
        return ''.join(result)
    
    def merge_adjacent_math(text):
        """Merge adjacent $...$ regions that should be together."""
        prev = None
        iterations = 0
        while prev != text and iterations < 50:
            prev = text
            iterations += 1
            
            # Merge $a$$b$ -> $a b$ (directly adjacent)
            text = re.sub(r'\$([^$]+)\$\$([^$]+)\$', r'$\1 \2$', text)
            
            # Merge $a$ = $b$ -> $a = b$ (with operators, keeping spacing)
            text = re.sub(r'\$([^$]+)\$\s*([=+\-*/])\s*\$([^$]+)\$', r'$\1 \2 \3$', text)
            
            # Merge $a$ $b$ -> $a b$ ONLY when both are pure math (LaTeX commands)
            # Don't merge when it would eat up text boundaries
            text = re.sub(r'\$([\\][^$]{1,40})\$\s+\$([\\][^$]{1,40})\$', r'$\1 \2$', text)
        
        return text
    
    def fix_nested_issues(text):
        """Fix issues with nested $ signs in braces."""
        
        # Fix patterns like {$a$} -> {a} (remove $ inside braces)
        def fix_braces(match):
            content = match.group(1)
            # Remove $ signs from inside braces
            content = content.replace('$', '')
            return '{' + content + '}'
        
        # Iteratively fix nested $ in braces
        prev = None
        while prev != text:
            prev = text
            text = re.sub(r'\{([^{}]*\$[^{}]*)\}', fix_braces, text)
        
        return text
    
    def cleanup(text):
        """Final cleanup."""
        # Fix internal spacing issues in math: "$ text $" -> "$text$"
        # But preserve intentional spaces like "$a + b$"
        
        # First, normalize spaces inside $...$ - remove leading/trailing spaces
        def fix_math_spaces(match):
            content = match.group(1)
            # Strip leading/trailing spaces
            content = content.strip()
            # Collapse multiple internal spaces
            content = re.sub(r'\s{2,}', ' ', content)
            return '$' + content + '$'
        
        text = re.sub(r'\$([^$]+)\$', fix_math_spaces, text)
        
        # Remove empty math: $$
        text = re.sub(r'\$\s*\$', '', text)
        
        # Fix double dollars: $$ -> $
        text = re.sub(r'\$\$+', '$', text)
        
        # Fix "e^$" patterns (exponential with broken $)
        text = re.sub(r'e\^\$\(([^)]+)\)\$', r'$e^{\1}$', text)
        text = re.sub(r'e\^\$([^$]+)\$', r'$e^{\1}$', text)
        
        # Fix standalone ^{...} or _{...} that should be merged with preceding letter
        text = re.sub(r'([a-zA-Z])\s*\$\^{([^}]+)}\$', r'$\1^{\2}$', text)
        text = re.sub(r'([a-zA-Z])\s*\$_{([^}]+)}\$', r'$\1_{\2}$', text)
        
        # Add space BEFORE opening $ when preceded by alphanumeric and NOT already a $
        # Pattern: letter/digit followed by $ that starts a math expression (has content before next $)
        # We need to be careful not to match the closing $ of an expression
        # Solution: only add space when the character before $ is alphanumeric AND the $ is followed by \
        text = re.sub(r'([a-zA-Z0-9])\$(\\)', r'\1 $\2', text)
        
        # Add space AFTER closing $ when followed by letter
        text = re.sub(r'\$([^$]+)\$([a-zA-Z])', r'$\1$ \2', text)
        
        # Remove space before punctuation after $
        text = re.sub(r'\$ ([,.\)\]:;])', r'$\1', text)
        
        # Collapse multiple spaces
        text = re.sub(r'  +', ' ', text)
        
        return text
    
    def process_text(text):
        """Apply all conversions."""
        text = strip_dollar_signs(text)
        text = wrap_math_expressions(text)
        text = merge_adjacent_math(text)
        text = fix_nested_issues(text)
        text = cleanup(text)
        return text
    
    def process_obj(obj):
        """Recursively process JSON object."""
        if isinstance(obj, dict):
            return {k: process_obj(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [process_obj(item) for item in obj]
        elif isinstance(obj, str):
            return process_text(obj)
        return obj
    
    # Process file
    line_num = 0
    with open(input_file, 'r', encoding='utf-8') as f_in:
        with open(output_file, 'w', encoding='utf-8') as f_out:
            for line_num, line in enumerate(f_in, 1):
                try:
                    obj = json.loads(line)
                    obj = process_obj(obj)
                    json.dump(obj, f_out, ensure_ascii=False)
                    f_out.write('\n')
                except Exception as e:
                    print(f"Error on line {line_num}: {e}")
                    import traceback
                    traceback.print_exc()
                    f_out.write(line)
    
    print(f"Processed {line_num} lines -> {output_file}")
    return output_file


# Test function
def test_conversion():
    """Test the conversion on some example strings."""
    test_cases = [
        # From your file - broken patterns
        (r'$\frac${k_{0;1-$\alpha$}}{n$\lambda_{obi}$}', r'$\frac{k_{0;1-\alpha}}{n\lambda_{obi}}$'),
        (r'$\int_0$^$\infty$', r'$\int_0^\infty$'),
        (r'$\beta_M$ LE', r'$\beta_M$ LE'),  # This one is tricky
        (r'e^{-$\lambda$ t}', r'$e^{-\lambda t}$'),
        (r'$\lambda$t', r'$\lambda t$'),
        (r'T_0', r'$T_0$'),
        (r'55^$\circ$ C', r'$55^\circ$ C'),
    ]
    
    for input_str, expected in test_cases:
        # Strip dollars first
        stripped = input_str.replace('$', '')
        print(f"Input:    {input_str}")
        print(f"Stripped: {stripped}")
        # Process would go here
        print()



In [5]:
fix_latex('../content/questions_dataset.jsonl', '../content/questions_dataset_cleaned.jsonl')

Processed 66 lines -> ../content/questions_dataset_cleaned.jsonl


'../content/questions_dataset_cleaned.jsonl'